In [1]:
# ==========================================================
# CREDIT RISK MODELING PIPELINE WITH SMOTE
# ==========================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from imblearn.over_sampling import SMOTE

# -----------------------------
# 1. CREATE PROJECT FOLDERS
# -----------------------------
folders = ["data", "models", "figures", "results"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

# -----------------------------
# 2. GENERATE SIMULATED CREDIT DATA
# -----------------------------
np.random.seed(42)
n = 50000

# Features: age, income, credit_score
mean = [45, 50000, 700]
cov = [[100, 2000, 50], [2000, 40000000, 500], [50, 500, 400]]
age_income_score = np.random.multivariate_normal(mean, cov, n)
age = np.clip(age_income_score[:,0], 18, 70)
income = np.clip(age_income_score[:,1], 20000, 200000)
credit_score = np.clip(age_income_score[:,2], 300, 850)

# Additional features
loan_amount = np.random.normal(15000, 5000, n)
loan_term = np.random.randint(6, 60, n)
credit_cards = np.random.randint(0, 5, n)
previous_defaults = np.random.randint(0, 3, n)

# PD signal (~10–20% defaults)
logit = -5 + 0.01*(age-45) - 0.00002*(income-50000) - 0.008*(credit_score-700) + 0.6*previous_defaults
pd_prob = 1 / (1 + np.exp(-logit))
PD = np.random.binomial(1, pd_prob)

# LGD signal
LGD = np.clip(0.2 + 0.5*(700-credit_score)/400 + 0.1*np.random.rand(n), 0.2, 0.8)

# EAD
EAD = np.random.uniform(5000, 20000, n)

# Create DataFrame
data = pd.DataFrame({
    "age": age,
    "income": income,
    "loan_amount": loan_amount,
    "loan_term": loan_term,
    "credit_cards": credit_cards,
    "previous_defaults": previous_defaults,
    "credit_score": credit_score,
    "PD": PD,
    "LGD": LGD,
    "EAD": EAD
})
data.to_csv("data/credit_data.csv", index=False)

# -----------------------------
# 3. TRAIN-TEST SPLIT
# -----------------------------
train, test = train_test_split(data, test_size=0.2, random_state=42)
features = ["age", "income", "loan_amount", "loan_term", "credit_cards", "previous_defaults", "credit_score"]

X_train, X_test = train[features], test[features]
y_train_pd, y_test_pd = train["PD"], test["PD"]
y_train_lgd, y_test_lgd = train["LGD"], test["LGD"]
EAD_test = np.array(test["EAD"])

# -----------------------------
# 4. BALANCE DATA WITH SMOTE
# -----------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train_pd)

# -----------------------------
# 5. TRAIN MULTIPLE PD MODELS
# -----------------------------
pd_models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=200),
    "GradientBoosting": GradientBoostingClassifier()
}

results = {}
print("\nTraining PD Models...\n")
for name, model in pd_models.items():
    model.fit(X_train_res, y_train_res)
    prob = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test_pd, prob)
    results[name] = {"model": model, "roc_auc": auc}
    print(f"{name} ROC-AUC: {auc:.4f}")

# -----------------------------
# 6. SELECT BEST PD MODEL
# -----------------------------
best_model_name = max(results, key=lambda x: results[x]["roc_auc"])
best_model = results[best_model_name]["model"]
best_auc = results[best_model_name]["roc_auc"]
print(f"\nBest PD Model: {best_model_name} | ROC-AUC: {best_auc:.4f}")
joblib.dump(best_model, "models/best_pd_model.pkl")

# -----------------------------
# 7. TRAIN LGD MODEL
# -----------------------------
lgd_model = RandomForestRegressor(n_estimators=200)
lgd_model.fit(X_train, y_train_lgd)
joblib.dump(lgd_model, "models/lgd_model.pkl")
lgd_pred = lgd_model.predict(X_test)

# -----------------------------
# 8. PD PREDICTIONS & METRICS
# -----------------------------
pd_prob = best_model.predict_proba(X_test)[:,1]
brier = brier_score_loss(y_test_pd, pd_prob)
hit_rate = np.mean((pd_prob > 0.5) == y_test_pd)
expected_loss = np.mean(pd_prob * lgd_pred * EAD_test)

# -----------------------------
# 9. STRESS SCENARIO & MONTE CARLO
# -----------------------------
stress_pd = np.clip(pd_prob * (1 + 0.5*(X_test["previous_defaults"] > 1)), 0, 1)
n_mc = 10000
portfolio_loss = []
stress_pd_array = stress_pd.to_numpy()
for _ in range(n_mc):
    sim_default = np.random.binomial(1, stress_pd_array)
    portfolio_loss.append(np.sum(sim_default * lgd_pred * EAD_test))
portfolio_loss = np.array(portfolio_loss)
stress_loss = portfolio_loss.mean()
portfolio_var = np.percentile(portfolio_loss, 99)

# -----------------------------
# PORTFOLIO LOSS DISTRIBUTION
# -----------------------------
plt.figure(figsize=(8,5))
plt.hist(portfolio_loss, bins=50)

plt.axvline(portfolio_var, linestyle="--", label="99% VaR")

plt.title("Monte Carlo Portfolio Loss Distribution")
plt.xlabel("Portfolio Loss")
plt.ylabel("Frequency")
plt.legend()

plt.tight_layout()
plt.savefig("figures/portfolio_loss_distribution.png")
plt.close()

# -----------------------------
# 10. BOOTSTRAP CONFIDENCE INTERVAL
# -----------------------------
boot_means = []
n_boot = 2000
for i in range(n_boot):
    idx = np.random.choice(len(EAD_test), len(EAD_test), replace=True)
    boot_means.append(np.mean(pd_prob[idx] * lgd_pred[idx] * EAD_test[idx]))
ci_low, ci_high = np.percentile(boot_means, [2.5,97.5])

# -----------------------------
# 11. SAVE METRICS
# -----------------------------
metrics = pd.DataFrame({
    "Metric": ["Best Model", "ROC-AUC", "Brier Score", "Hit Rate", "Expected Loss",
               "Stress Loss", "Portfolio VaR", "CI Lower", "CI Upper"],
    "Value": [best_model_name, best_auc, brier, hit_rate, expected_loss,
              stress_loss, portfolio_var, ci_low, ci_high]
})
metrics.to_csv("results/model_metrics.csv", index=False)
print("\nQuantitative Metrics\n", metrics)

# -----------------------------
# 12. VISUALIZATIONS
# -----------------------------
plt.figure()
plt.hist(pd_prob, bins=50)
plt.title("Predicted Probability of Default")
plt.xlabel("PD"); plt.ylabel("Frequency")
plt.savefig("figures/pd_histogram.png"); plt.close()

plt.figure()
plt.scatter(y_test_lgd, lgd_pred, alpha=0.3)
plt.title("Predicted vs Actual LGD")
plt.xlabel("Actual LGD"); plt.ylabel("Predicted LGD")
plt.savefig("figures/lgd_scatter.png"); plt.close()

plt.figure()
plt.hist(boot_means, bins=50)
plt.axvline(ci_low, color='r')
plt.axvline(ci_high, color='r')
plt.title("Bootstrap Expected Loss Distribution")
plt.savefig("figures/bootstrap_ci.png"); plt.close()

# -----------------------------
# 13. FEATURE IMPORTANCE FOR LOGISTIC REGRESSION
# -----------------------------
if best_model_name == "LogisticRegression":
    importance = pd.Series(best_model.coef_[0], index=features).sort_values(ascending=False)
    plt.figure(figsize=(8,5))
    importance.plot(kind='barh')
    plt.title("Logistic Regression Feature Importance")
    plt.xlabel("Coefficient Value")
    plt.tight_layout()
    plt.savefig("figures/logreg_feature_importance.png")
    plt.close()
    print("Feature importance plot saved in figures/logreg_feature_importance.png")

# -----------------------------
# 14. FINAL REPORT
# -----------------------------
report = (
    f"CREDIT RISK MODELING REPORT\n\n"
    f"Dataset Size: {n} observations\n"
    f"Best PD Model: {best_model_name}\n"
    f"ROC-AUC: {best_auc:.4f}\n"
    f"Brier Score: {brier:.4f}\n"
    f"Hit Rate: {hit_rate:.4f}\n"
    f"Expected Portfolio Loss: {expected_loss:.2f}\n"
    f"Stress Portfolio Loss: {stress_loss:.2f}\n"
    f"Portfolio VaR (99%): {portfolio_var:.2f}\n"
    f"95% Confidence Interval for Expected Loss: [{ci_low:.2f}, {ci_high:.2f}]\n\n"
    f"Outputs Generated:\n"
    f"• Model files in /models\n"
    f"• Figures in /figures\n"
    f"• Metrics in /results\n"
)

with open("results/model_report.txt", "w") as f:
    f.write(report)

print("\nProject completed successfully.")
print("Outputs saved to folders: data / figures / results")


Training PD Models...

LogisticRegression ROC-AUC: 0.6162
RandomForest ROC-AUC: 0.5643
GradientBoosting ROC-AUC: 0.5589

Best PD Model: LogisticRegression | ROC-AUC: 0.6162

Quantitative Metrics
           Metric               Value
0     Best Model  LogisticRegression
1        ROC-AUC            0.616226
2    Brier Score            0.237595
3       Hit Rate              0.5872
4  Expected Loss         1478.206484
5    Stress Loss     17502970.019291
6  Portfolio VaR     17845814.761366
7       CI Lower          1466.16257
8       CI Upper         1490.404341
Feature importance plot saved in figures/logreg_feature_importance.png

Project completed successfully.
Outputs saved to folders: data / figures / results
